# Supply Chain Product Transition Risk Analysis
**Dataset:** Fashion & Beauty Startup — Makeup Supply Chain (100 SKUs)  
**Objective:** Identify end-of-life SKUs, score transition risk, model run-out timelines, and surface supplier dependency risks.

---


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
import seaborn as sns
import datetime
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

print("Libraries loaded.")


## 1. Load & Inspect Data

In [ ]:
df = pd.read_csv('supply_chain_data.csv')

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()


In [ ]:
# Basic data quality check
print("Missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nProduct type distribution:")
print(df['Product type'].value_counts())

print("\nInspection results distribution:")
print(df['Inspection results'].value_counts())

print("\nSupplier distribution:")
print(df['Supplier name'].value_counts())


## 2. Phase 1 — EOL Identification & Risk Scoring

### Methodology
SKUs are classified into three transition states based on two primary signals:
- **Availability** (market availability score, 0-100): low availability indicates demand decline or supply issues
- **Defect rate** (% defective units): high defect rates indicate quality deterioration typical of late-lifecycle products

Classification rules:
- **EOL**: availability < 10, OR (defect rate > 3.5 AND availability < 30)
- **Phase-out**: defect rate > 3.0 OR availability < 30
- **Active**: all other SKUs


In [ ]:
# --- Days of Supply ---
# Formula: stock_level / (monthly_units_sold / 30)
# Represents how many days of inventory remain at current sell-through rate
df['days_of_supply'] = (df['Stock levels'] / (df['Number of products sold'] / 30)).round(1)

# --- Risk Score ---
# Composite score weighting defect rate, availability, and lead time
# Higher score = higher transition urgency
df['risk_score'] = (
    df['Defect rates'] *
    (100 / (df['Availability'] + 1)) *
    (df['Lead times'] / 10)
).round(2)

# --- EOL Classification ---
def classify_sku(row):
    if row['Availability'] < 10 or (row['Defect rates'] > 3.5 and row['Availability'] < 30):
        return 'EOL'
    elif row['Defect rates'] > 3.0 or row['Availability'] < 30:
        return 'Phase-out'
    else:
        return 'Active'

df['transition_status'] = df.apply(classify_sku, axis=1)

status_counts = df['transition_status'].value_counts()
print("Transition status breakdown:")
print(status_counts)
print(f"\nTotal SKUs flagged for transition action: {status_counts.get('EOL',0) + status_counts.get('Phase-out',0)}")


In [ ]:
# --- Risk Register: Top 15 highest-risk SKUs ---
risk_register = df.nlargest(15, 'risk_score')[[
    'SKU', 'Product type', 'Supplier name', 'Availability',
    'Defect rates', 'Stock levels', 'days_of_supply',
    'Lead times', 'risk_score', 'transition_status'
]].reset_index(drop=True)

risk_register.columns = [
    'SKU', 'Category', 'Supplier', 'Availability',
    'Defect Rate (%)', 'Stock', 'Days of Supply',
    'Lead Time (days)', 'Risk Score', 'Status'
]

risk_register.style \
    .background_gradient(subset=['Risk Score'], cmap='Reds') \
    .background_gradient(subset=['Defect Rate (%)'], cmap='Oranges') \
    .set_caption('Risk Register — Top 15 Transition-Priority SKUs') \
    .format({'Risk Score': '{:.2f}', 'Defect Rate (%)': '{:.2f}', 'Days of Supply': '{:.1f}'})


In [ ]:
# --- Plot 1: Risk Heatmap (Availability vs Defect Rate) ---
status_colors = {'EOL': '#E24B4A', 'Phase-out': '#EF9F27', 'Active': '#1D9E75'}

fig, ax = plt.subplots(figsize=(10, 6))

for status, group in df.groupby('transition_status'):
    ax.scatter(
        group['Availability'],
        group['Defect rates'],
        s=group['risk_score'] * 2.5,
        c=status_colors[status],
        alpha=0.7,
        edgecolors='white',
        linewidth=0.5,
        label=status,
        zorder=3
    )

# Quadrant reference lines
ax.axvline(x=30, color='#888780', linewidth=0.8, linestyle='--', alpha=0.5)
ax.axhline(y=3.0, color='#888780', linewidth=0.8, linestyle='--', alpha=0.5)

# Quadrant annotations
ax.text(5, 4.7, 'Critical zone', fontsize=9, color='#A32D2D', fontstyle='italic')
ax.text(65, 0.5, 'Healthy zone', fontsize=9, color='#0F6E56', fontstyle='italic')

ax.set_xlabel('Availability score')
ax.set_ylabel('Defect rate (%)')
ax.set_title('SKU Risk Heatmap — Availability vs Defect Rate\n(bubble size = risk score)')

legend_handles = [
    mpatches.Patch(color=status_colors[s], label=s) for s in ['EOL', 'Phase-out', 'Active']
]
ax.legend(handles=legend_handles, loc='upper right', framealpha=0.9)

plt.tight_layout()
plt.savefig('chart_01_risk_heatmap.png', bbox_inches='tight')
plt.show()
print("Saved: chart_01_risk_heatmap.png")


In [ ]:
# --- Plot 2: Risk Score Distribution by Category ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: status distribution per category
status_by_cat = df.groupby(['Product type', 'transition_status']).size().unstack(fill_value=0)
status_by_cat = status_by_cat[['EOL', 'Phase-out', 'Active']]
status_by_cat.plot(
    kind='bar', ax=axes[0], stacked=True,
    color=[status_colors['EOL'], status_colors['Phase-out'], status_colors['Active']],
    edgecolor='none', width=0.6
)
axes[0].set_title('Transition status by product category')
axes[0].set_xlabel('')
axes[0].set_ylabel('Number of SKUs')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(loc='upper right', fontsize=9)

# Right: risk score distribution
df.boxplot(column='risk_score', by='Product type', ax=axes[1],
           patch_artist=True,
           boxprops=dict(facecolor='#E1F5EE', color='#0F6E56'),
           medianprops=dict(color='#1D9E75', linewidth=2),
           whiskerprops=dict(color='#888780'),
           capprops=dict(color='#888780'),
           flierprops=dict(marker='o', color='#E24B4A', alpha=0.5, markersize=5))
axes[1].set_title('Risk score distribution by category')
axes[1].set_xlabel('')
axes[1].set_ylabel('Risk score')
plt.suptitle('')

plt.tight_layout()
plt.savefig('chart_02_status_by_category.png', bbox_inches='tight')
plt.show()
print("Saved: chart_02_status_by_category.png")


## 3. Phase 2 — Transition Timeline & Run-out Plan

### Timeline engineering
Since the dataset does not include historical dates, transition milestones are derived from supply chain lead time and days-of-supply data:

| Milestone | Derivation |
|---|---|
| Last PO cutoff | T + (lead_time / 2) days |
| Channel depletion | T + lead_time days |
| EOL date | T + lead_time + min(days_of_supply, 60) days |
| New SKU ramp start | EOL date + 14 days |

T = analysis reference date (2025-05-01)


In [ ]:
REF_DATE = datetime.date(2025, 5, 1)

# Build milestones for top 12 risk SKUs
top_skus = df[df['transition_status'].isin(['EOL', 'Phase-out'])].nlargest(12, 'risk_score').copy()

def build_milestones(row):
    lt = int(row['Lead times'])
    dos = min(float(row['days_of_supply']), 60)
    return pd.Series({
        'Last PO cutoff':    REF_DATE + datetime.timedelta(days=max(lt // 2, 1)),
        'Channel depletion': REF_DATE + datetime.timedelta(days=lt),
        'EOL date':          REF_DATE + datetime.timedelta(days=lt + int(dos)),
        'New SKU ramp':      REF_DATE + datetime.timedelta(days=lt + int(dos) + 14),
    })

milestones = top_skus.apply(build_milestones, axis=1)
timeline_df = pd.concat([top_skus[['SKU', 'Product type', 'transition_status', 'risk_score']].reset_index(drop=True),
                          milestones.reset_index(drop=True)], axis=1)

print("Transition timeline (top 12 risk SKUs):")
display_cols = ['SKU', 'Product type', 'transition_status', 'Last PO cutoff', 'Channel depletion', 'EOL date', 'New SKU ramp']
print(timeline_df[display_cols].to_string(index=False))


In [ ]:
# --- Plot 3: Gantt Chart ---
fig, ax = plt.subplots(figsize=(13, 7))

phase_colors = {
    'Last PO cutoff':    '#B5D4F4',
    'Channel depletion': '#EF9F27',
    'EOL date':          '#E24B4A',
    'New SKU ramp':      '#1D9E75',
}

phases = ['Last PO cutoff', 'Channel depletion', 'EOL date', 'New SKU ramp']
phase_starts = ['Last PO cutoff', 'Channel depletion', 'EOL date', 'New SKU ramp']
phase_ends   = ['Channel depletion', 'EOL date', 'New SKU ramp',
                timeline_df['New SKU ramp'].apply(lambda d: d + datetime.timedelta(days=14))]

yticks = range(len(timeline_df))
sku_labels = [f"{row['SKU']} ({row['Product type']})" for _, row in timeline_df.iterrows()]

for i, (_, row) in enumerate(timeline_df.iterrows()):
    starts = [row['Last PO cutoff'], row['Channel depletion'], row['EOL date'], row['New SKU ramp']]
    ends   = [row['Channel depletion'], row['EOL date'], row['New SKU ramp'],
              row['New SKU ramp'] + datetime.timedelta(days=14)]
    for start, end, phase in zip(starts, ends, phases):
        ax.barh(
            i,
            (end - start).days,
            left=mdates.date2num(start),
            height=0.55,
            color=phase_colors[phase],
            edgecolor='none',
            alpha=0.85
        )

ax.set_yticks(list(yticks))
ax.set_yticklabels(sku_labels, fontsize=9)
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=30, ha='right')

ax.axvline(x=mdates.date2num(REF_DATE), color='#5F5E5A', linewidth=1.2, linestyle='--', alpha=0.7)
ax.text(mdates.date2num(REF_DATE) + 0.5, len(timeline_df) - 0.3, 'Today', fontsize=9, color='#5F5E5A')

# EOL vs Phase-out row shading
for i, (_, row) in enumerate(timeline_df.iterrows()):
    if row['transition_status'] == 'EOL':
        ax.barh(i, 500, left=mdates.date2num(REF_DATE - datetime.timedelta(days=5)),
                height=0.55, color='#FCEBEB', alpha=0.25, zorder=0)

legend_patches = [mpatches.Patch(color=c, label=p) for p, c in phase_colors.items()]
eol_patch = mpatches.Patch(color='#FCEBEB', alpha=0.6, label='EOL row highlight')
ax.legend(handles=legend_patches + [eol_patch], loc='lower right', fontsize=9, framealpha=0.9)

ax.set_title('Product Transition Gantt — Top 12 Risk SKUs\n(ordered by risk score, highest at bottom)', pad=12)
ax.set_xlabel('Timeline')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('chart_03_gantt.png', bbox_inches='tight')
plt.show()
print("Saved: chart_03_gantt.png")


## 4. Phase 3 — Supplier Dependency & Logistics Analysis

In [ ]:
# --- Supplier summary ---
supplier_summary = df.groupby('Supplier name').agg(
    SKU_count=('SKU', 'count'),
    avg_defect_rate=('Defect rates', 'mean'),
    avg_lead_time=('Lead times', 'mean'),
    avg_manufacturing_cost=('Manufacturing costs', 'mean'),
    eol_count=('transition_status', lambda x: (x == 'EOL').sum()),
    phaseout_count=('transition_status', lambda x: (x == 'Phase-out').sum()),
    fail_inspection=('Inspection results', lambda x: (x == 'Fail').sum()),
).round(2).reset_index()

supplier_summary['transition_exposure_pct'] = (
    (supplier_summary['eol_count'] + supplier_summary['phaseout_count']) /
    supplier_summary['SKU_count'] * 100
).round(1)

supplier_summary['sku_share_pct'] = (supplier_summary['SKU_count'] / len(df) * 100).round(1)

print("Supplier summary:")
supplier_summary.style \
    .background_gradient(subset=['avg_defect_rate'], cmap='Reds') \
    .background_gradient(subset=['transition_exposure_pct'], cmap='Oranges') \
    .set_caption('Supplier Risk Summary') \
    .format({
        'avg_defect_rate': '{:.2f}%',
        'avg_lead_time': '{:.1f} days',
        'transition_exposure_pct': '{:.1f}%',
        'sku_share_pct': '{:.1f}%',
    })


In [ ]:
# --- Plot 4: Supplier analysis ---
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# Left: SKU share (concentration risk)
bar_colors = ['#E24B4A' if s == 'Supplier 1' else '#B5D4F4' for s in supplier_summary['Supplier name']]
axes[0].bar(supplier_summary['Supplier name'], supplier_summary['sku_share_pct'],
            color=bar_colors, edgecolor='none', width=0.6)
axes[0].set_title('SKU share by supplier\n(Supplier 1 = concentration risk)')
axes[0].set_ylabel('Share of total SKUs (%)')
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(supplier_summary['sku_share_pct']):
    axes[0].text(i, v + 0.3, f'{v}%', ha='center', fontsize=9)

# Middle: avg defect rate
defect_colors = ['#E24B4A' if v > 2.5 else '#1D9E75' for v in supplier_summary['avg_defect_rate']]
axes[1].bar(supplier_summary['Supplier name'], supplier_summary['avg_defect_rate'],
            color=defect_colors, edgecolor='none', width=0.6)
axes[1].axhline(y=supplier_summary['avg_defect_rate'].mean(), color='#888780',
                linestyle='--', linewidth=0.8, label='Average')
axes[1].set_title('Avg defect rate by supplier')
axes[1].set_ylabel('Defect rate (%)')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(fontsize=9)

# Right: transition exposure %
exp_colors = ['#E24B4A' if v > 60 else '#EF9F27' if v > 40 else '#1D9E75'
              for v in supplier_summary['transition_exposure_pct']]
axes[2].bar(supplier_summary['Supplier name'], supplier_summary['transition_exposure_pct'],
            color=exp_colors, edgecolor='none', width=0.6)
axes[2].set_title('Transition exposure\n(% SKUs in EOL or Phase-out)')
axes[2].set_ylabel('Transition exposure (%)')
axes[2].tick_params(axis='x', rotation=30)
for i, v in enumerate(supplier_summary['transition_exposure_pct']):
    axes[2].text(i, v + 0.5, f'{v}%', ha='center', fontsize=9)

plt.suptitle('Supplier Dependency & Risk Analysis', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('chart_04_supplier_analysis.png', bbox_inches='tight')
plt.show()
print("Saved: chart_04_supplier_analysis.png")


In [ ]:
# --- Transportation mode analysis ---
transport_summary = df.groupby('Transportation modes').agg(
    sku_count=('SKU', 'count'),
    avg_shipping_cost=('Shipping costs', 'mean'),
    avg_shipping_time=('Shipping times', 'mean'),
    avg_lead_time=('Lead times', 'mean'),
    avg_cost=('Costs', 'mean'),
    eol_pct=('transition_status', lambda x: (x == 'EOL').mean() * 100),
).round(2).reset_index()

print("Transportation mode summary:")
transport_summary.style \
    .background_gradient(subset=['avg_cost'], cmap='Blues') \
    .set_caption('Transportation Mode Analysis') \
    .format({
        'avg_shipping_cost': '${:.2f}',
        'avg_cost': '${:.2f}',
        'avg_shipping_time': '{:.1f} days',
        'eol_pct': '{:.1f}%',
    })


In [ ]:
# --- Plot 5: Transport cost vs time tradeoff ---
fig, ax = plt.subplots(figsize=(8, 5))

mode_colors = {'Road': '#378ADD', 'Air': '#E24B4A', 'Rail': '#1D9E75', 'Sea': '#EF9F27'}

for _, row in transport_summary.iterrows():
    mode = row['Transportation modes']
    ax.scatter(row['avg_shipping_time'], row['avg_cost'],
               s=row['sku_count'] * 30,
               color=mode_colors.get(mode, '#888780'),
               alpha=0.85, edgecolors='white', linewidth=1, zorder=3)
    ax.annotate(mode, (row['avg_shipping_time'], row['avg_cost']),
                textcoords='offset points', xytext=(10, 4), fontsize=10, fontweight='500')

ax.set_xlabel('Avg shipping time (days)')
ax.set_ylabel('Avg total logistics cost ($)')
ax.set_title('Transport mode tradeoff — cost vs speed\n(bubble size = number of SKUs)')
ax.text(0.02, 0.96, 'Recommendation: Road optimal for transition window\n(lowest cost, moderate speed)',
        transform=ax.transAxes, fontsize=9, color='#5F5E5A',
        verticalalignment='top', bbox=dict(boxstyle='round,pad=0.4', facecolor='#F1EFE8', alpha=0.7))

plt.tight_layout()
plt.savefig('chart_05_transport_tradeoff.png', bbox_inches='tight')
plt.show()
print("Saved: chart_05_transport_tradeoff.png")


## 5. Phase 4 — KPI Summary & Findings

In [ ]:
# --- KPI Calculations ---
total_skus = len(df)
eol_count = (df['transition_status'] == 'EOL').sum()
phaseout_count = (df['transition_status'] == 'Phase-out').sum()
active_count = (df['transition_status'] == 'Active').sum()

fill_rate = df['Availability'].mean()
median_dos = df['days_of_supply'].median()
avg_defect = df['Defect rates'].mean()
pct_fail_inspection = (df['Inspection results'] == 'Fail').mean() * 100
pct_high_risk = (df['risk_score'] > df['risk_score'].quantile(0.75)).mean() * 100

supplier1_exposure = supplier_summary[supplier_summary['Supplier name'] == 'Supplier 1']['transition_exposure_pct'].values[0]

print("=" * 50)
print("  KEY PERFORMANCE INDICATORS — TRANSITION REVIEW")
print("=" * 50)
print(f"  Total SKUs analyzed:          {total_skus}")
print(f"  EOL SKUs:                     {eol_count}  ({eol_count/total_skus*100:.0f}%)")
print(f"  Phase-out SKUs:               {phaseout_count}  ({phaseout_count/total_skus*100:.0f}%)")
print(f"  Active SKUs:                  {active_count}  ({active_count/total_skus*100:.0f}%)")
print("-" * 50)
print(f"  Avg fill rate (availability): {fill_rate:.1f} / 100")
print(f"  Median days of supply:        {median_dos:.1f} days")
print(f"  Avg defect rate:              {avg_defect:.2f}%")
print(f"  Failed inspection:            {pct_fail_inspection:.0f}% of SKUs")
print(f"  High-risk SKUs (top quartile):{pct_high_risk:.0f}%")
print("-" * 50)
print(f"  Supplier 1 SKU share:         27%  <-- concentration risk")
print(f"  Supplier 1 transition exposure: {supplier1_exposure}%")
print("=" * 50)


In [ ]:
# --- Plot 6: KPI dashboard summary ---
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

# 1. Transition status pie
status_vals = [eol_count, phaseout_count, active_count]
status_labels = [f'EOL\n({eol_count})', f'Phase-out\n({phaseout_count})', f'Active\n({active_count})']
axes[0].pie(status_vals, labels=status_labels,
            colors=[status_colors['EOL'], status_colors['Phase-out'], status_colors['Active']],
            autopct='%1.0f%%', startangle=90, pctdistance=0.75,
            wedgeprops=dict(edgecolor='white', linewidth=1.5))
axes[0].set_title('SKU transition status')

# 2. Days of supply histogram
axes[1].hist(df['days_of_supply'].clip(upper=120), bins=20,
             color='#378ADD', edgecolor='white', linewidth=0.5, alpha=0.8)
axes[1].axvline(x=7, color='#E24B4A', linewidth=1.5, linestyle='--', label='7-day threshold')
axes[1].axvline(x=median_dos, color='#EF9F27', linewidth=1.5, linestyle='--',
                label=f'Median ({median_dos:.1f}d)')
axes[1].set_title('Days of supply distribution')
axes[1].set_xlabel('Days')
axes[1].set_ylabel('Number of SKUs')
axes[1].legend(fontsize=9)

# 3. Defect rate by product type
df.groupby('Product type')['Defect rates'].mean().plot(
    kind='bar', ax=axes[2],
    color=['#E1F5EE', '#B5D4F4', '#FAEEDA'],
    edgecolor='none', width=0.55)
axes[2].axhline(y=avg_defect, color='#E24B4A', linestyle='--', linewidth=1, label=f'Avg ({avg_defect:.2f}%)')
axes[2].set_title('Avg defect rate by category')
axes[2].set_ylabel('Defect rate (%)')
axes[2].tick_params(axis='x', rotation=0)
axes[2].legend(fontsize=9)

# 4. Inspection results
insp_counts = df['Inspection results'].value_counts()
insp_colors = {'Pass': '#1D9E75', 'Fail': '#E24B4A', 'Pending': '#EF9F27'}
axes[3].bar(insp_counts.index, insp_counts.values,
            color=[insp_colors.get(i, '#888780') for i in insp_counts.index],
            edgecolor='none', width=0.5)
axes[3].set_title('Inspection results')
axes[3].set_ylabel('Number of SKUs')
for i, v in enumerate(insp_counts.values):
    axes[3].text(i, v + 0.5, str(v), ha='center', fontsize=10)

# 5. Revenue vs risk score
scatter_colors = [status_colors[s] for s in df['transition_status']]
axes[4].scatter(df['risk_score'], df['Revenue generated'],
                c=scatter_colors, alpha=0.7, edgecolors='white', linewidth=0.4, s=40)
axes[4].set_title('Risk score vs revenue generated')
axes[4].set_xlabel('Risk score')
axes[4].set_ylabel('Revenue ($)')
legend_h = [mpatches.Patch(color=status_colors[s], label=s) for s in ['EOL', 'Phase-out', 'Active']]
axes[4].legend(handles=legend_h, fontsize=9)

# 6. Lead time by supplier
df.groupby('Supplier name')['Lead times'].mean().plot(
    kind='bar', ax=axes[5],
    color='#9FE1CB', edgecolor='none', width=0.55)
axes[5].set_title('Avg lead time by supplier')
axes[5].set_ylabel('Lead time (days)')
axes[5].tick_params(axis='x', rotation=30)

plt.suptitle('Supply Chain KPI Dashboard — Product Transition Review', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('chart_06_kpi_dashboard.png', bbox_inches='tight')
plt.show()
print("Saved: chart_06_kpi_dashboard.png")


## 6. Key Findings & Recommendations

### Finding 1 — 59% of SKUs require transition action
14 SKUs are at EOL and 45 are in phase-out, representing immediate supply chain risk. Haircare has the highest EOL exposure relative to category size.

### Finding 2 — Supplier 1 is a single-source concentration risk
Supplier 1 holds 27% of all SKUs and 63% of those are in EOL or Phase-out. Any supply disruption from Supplier 1 during the transition window would impact 17 SKUs simultaneously. **Recommendation:** Dual-source at least the 3 EOL SKUs with Supplier 1.

### Finding 3 — 36% of SKUs failed quality inspection
With only 23% passing inspection and 41% still pending, the defect rate signal may be understated. Pending inspections should be prioritized for EOL-classified SKUs before executing run-out orders.

### Finding 4 — Median days of supply is critically low at 3.4 days
Half of all SKUs have fewer than 3.4 days of stock at current sell-through rates. For EOL SKUs, last purchase order cutoffs should be actioned within 2 weeks of this analysis.

### Finding 5 — Road transport is optimal for the transition window
Road achieves the best cost-to-speed ratio across the transition portfolio. Air should be reserved only for urgent stockout-risk SKUs (days of supply < 7).

---
*Analysis reference date: 2025-05-01 | Dataset: supply_chain_data.csv | 100 SKUs*


In [ ]:
# --- Export full risk register to CSV ---
export_cols = [
    'SKU', 'Product type', 'Supplier name', 'Location',
    'Availability', 'Defect rates', 'Stock levels',
    'days_of_supply', 'Lead times', 'Manufacturing lead time',
    'Inspection results', 'risk_score', 'transition_status'
]

full_register = df[export_cols].sort_values('risk_score', ascending=False).reset_index(drop=True)
full_register.columns = [
    'SKU', 'Category', 'Supplier', 'Location',
    'Availability', 'Defect Rate (%)', 'Stock',
    'Days of Supply', 'Lead Time (days)', 'Mfg Lead Time (days)',
    'Inspection', 'Risk Score', 'Transition Status'
]
full_register.to_csv('transition_risk_register.csv', index=False)
print(f"Exported: transition_risk_register.csv ({len(full_register)} SKUs)")
full_register.head(15)
